In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Monitorar ou cancelar um job

Veja a lista das suas cargas de trabalho na [página Workloads](https://quantum.cloud.ibm.com/workloads).

## Ver o status do job
Acesse sua [tabela de Workloads](https://quantum.cloud.ibm.com/workloads) e verifique a coluna Status para saber se um job foi concluído ou falhou.

## Ver o uso restante
Acesse sua [tabela de Instâncias](https://quantum.cloud.ibm.com/instances) e selecione a aba correspondente ao plano para o qual você deseja ver o uso restante. O tempo total utilizado e o tempo total restante no seu plano são exibidos.

## Ver métricas sobre o número de jobs e cargas de trabalho enviados
Acesse a [página de Analytics](https://quantum.cloud.ibm.com/analytics) para ver o número total de jobs enviados, bem como a contagem de cargas de trabalho em batch e em sessão. Observe que você só consegue ver a página de Analytics para contas que você possui ou gerencia.

## Monitorar um job
Use a instância do job para verificar o status do job ou recuperar os resultados chamando o comando apropriado:

|                               |                                                                                                                                                                                                                             |
| ----------------------------- | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| job.result()                  | Consulta os resultados do job imediatamente após sua conclusão. Os resultados ficam disponíveis após o término do job. Por isso, `job.result()` é uma chamada bloqueante até que o job seja concluído.                      |
| job.job\_id()                 | Retorna o ID que identifica exclusivamente aquele job. Para recuperar os resultados posteriormente é necessário o ID do job. Por isso, recomenda-se salvar os IDs dos jobs que você possa querer recuperar mais tarde.       |
| job.status()                  | Verifica o status do job.                                                                                                                                                                                                   |
| job = service.job(\<job\_id>) | Recupera um job enviado anteriormente. Essa chamada requer o ID do job.                                                                                                                                                     |

<span id="retrieve-later"></span>
## Recuperar os resultados de um job posteriormente
Chame `service.job(\<job\_id>)` para recuperar um job enviado anteriormente. Se você não tiver o ID do job, ou quiser recuperar vários jobs de uma vez — incluindo jobs de QPUs (unidades de processamento quântico) desativadas —, chame `service.jobs()` com filtros opcionais. Consulte [QiskitRuntimeService.jobs](../api/qiskit-ibm-runtime/qiskit-runtime-service#jobs).

> **Note:** `service.jobs()` também retorna jobs executados pelo pacote `qiskit-ibm-provider`, que está descontinuado. Jobs enviados pelo pacote mais antigo (também descontinuado) `qiskit-ibmq-provider` não estão mais disponíveis.

### Exemplo
Este exemplo retorna os 10 jobs de runtime mais recentes executados em `my_backend`:

In [1]:
# This cell is hidden from users
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
import numpy as np


my_backend = "ibm_torino"
service = QiskitRuntimeService()
# backend = service.backend(my_backend)
backend = service.least_busy()

# Define two circuits, each with one parameter with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.cx(0, 1)
circuit.h(0)
circuit.measure_all()


pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)

params = np.random.uniform(size=(2, 3)).T

sampler_pub = (transpiled_circuit, params)

# Instantiate the new estimator object, then run the transpiled circuit
# using the set of parameters and observables.
sampler = SamplerV2(mode=backend)
job = sampler.run([sampler_pub], shots=4)
print(job.job_id())

d305ck0ocacs73ajagvg


In [2]:
result = job.result()


spans = job.result().metadata["execution"]["execution_spans"]
print(spans)

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])


In [3]:
params = np.random.uniform(size=(2, 3))
params

array([[0.2260416 , 0.8747859 , 0.44361995],
       [0.94700856, 0.96826017, 0.98426562]])

In [4]:
mask = spans[0].mask(0)
mask

array([[[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]]])

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Initialize the account first.
service = QiskitRuntimeService()
# Use `limit` to retrieve a specific number of jobs. The default `limit` is 10.
service.jobs(backend_name=my_backend)

## Cancel a job

You can cancel a job from the IBM Quantum Platform dashboard either on the Workloads page or the details page for a specific workload. On the Workloads page, click the overflow menu at the end of the row for that workload, and select Cancel. If you are on the details page for a specific workload, use the Actions dropdown at the top of the page, and select Cancel.

In Qiskit, use `job.cancel()` to cancel a job.

<span id="execution-spans"></span>
## View Sampler execution spans

The results of [`SamplerV2`](/docs/api/qiskit-ibm-runtime/sampler-v2) jobs executed in Qiskit Runtime contain execution timing information in their metadata.
This timing information can be used to place upper and lower timestamp bounds on when particular shots were executed on the QPU.
Shots are grouped into [`ExecutionSpan`](/docs/api/qiskit-ibm-runtime/execution-span-execution-span) objects, each of which indicates a start time, a stop time, and a specification of which shots were collected in the span.

An execution span specifies which data was executed during its window by providing an [`ExecutionSpan.mask`](/docs/api/qiskit-ibm-runtime/execution-span-execution-span#mask) method. This method, given any [Primitive Unified Block (PUB)](/docs/guides/primitive-input-output) index, returns a boolean mask that is `True` for all shots executed during its window. PUBs are indexed by the order in which they were given to the Sampler run call. If, for example, a PUB has shape `(2, 3)` and was run with four shots, then the mask's shape is `(2, 3, 4)`. See the [execution_span](/docs/api/qiskit-ibm-runtime/execution-span) API page for full details.

Example:

To view execution span information, review the metadata of the result returned by `SamplerV2`, which comes in the form of an `ExecutionSpans` object. This object is a list-like container containing instances of subclasses of `ExecutionSpan`, such as `SliceSpan`:

In [6]:
from qiskit.primitives import BitArray

# Get the mask of the 1st PUB for the 0th span.
mask = spans[0].mask(0)

# Decide whether the 0th shot of parameter set (1, 2) occurred in this span.
in_this_span = mask[2, 1, 0]

# Create a new bit array containing only the PUB-1 data collected during this span.
bits = result[0].data.meas
filtered_data = BitArray(bits.array[mask], bits.num_bits)

Execution spans can be filtered to include information pertaining to specific PUBs, selected by their indices:

In [7]:
# take the subset of spans that reference data in PUBs 0 or 2
spans.filter_by_pub([0, 2])

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])

View global information about the collection of execution spans:

In [8]:
print("Number of execution spans:", len(spans))
print("  Start of the first span:", spans.start)
print("     End of the last span:", spans.stop)
print("       Total duration (s):", spans.duration)

Number of execution spans: 1
  Start of the first span: 2025-09-09 16:31:16.320568
     End of the last span: 2025-09-09 16:31:16.865858
       Total duration (s): 0.54529


Extract and inspect a particular span:

In [9]:
spans.sort()
print(" Start of first span:", spans[0].start)
print("   End of first span:", spans[0].stop)
print("#shots in first span:", spans[0].size)

 Start of first span: 2025-09-09 16:31:16.320568
   End of first span: 2025-09-09 16:31:16.865858
#shots in first span: 24


## Cancelar um job
Você pode cancelar um job pelo painel do IBM Quantum Platform, seja na página Workloads ou na página de detalhes de uma carga de trabalho específica. Na página Workloads, clique no menu de opções no final da linha daquela carga de trabalho e selecione Cancelar. Se estiver na página de detalhes de uma carga de trabalho específica, use o menu suspenso Ações no topo da página e selecione Cancelar.

No Qiskit, use `job.cancel()` para cancelar um job.

<span id="execution-spans"></span>
## Ver os spans de execução do Sampler
Os resultados dos jobs do [`SamplerV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2) executados no Qiskit Runtime contêm informações de temporização de execução em seus metadados.
Essas informações de temporização podem ser usadas para definir limites de timestamp superiores e inferiores sobre quando determinados shots foram executados na QPU.
Os shots são agrupados em objetos [`ExecutionSpan`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span-execution-span), cada um indicando um horário de início, um horário de término e uma especificação de quais shots foram coletados no span.

Um span de execução especifica quais dados foram executados durante sua janela por meio de um método [`ExecutionSpan.mask`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span-execution-span#mask). Esse método, dado o índice de qualquer [Primitive Unified Block (PUB)](/guides/primitive-input-output), retorna uma máscara booleana que é `True` para todos os shots executados durante sua janela. Os PUBs são indexados pela ordem em que foram passados para a chamada de execução do Sampler. Se, por exemplo, um PUB tem forma `(2, 3)` e foi executado com quatro shots, a forma da máscara será `(2, 3, 4)`. Consulte a página de API [execution_span](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span) para detalhes completos.

Exemplo:
Para ver as informações de span de execução, consulte os metadados do resultado retornado pelo `SamplerV2`, que aparecem na forma de um objeto `ExecutionSpans`. Esse objeto é um contêiner semelhante a uma lista que contém instâncias de subclasses de `ExecutionSpan`, como `SliceSpan`: